In [ ]:
# Colab Cell 1 — Install dependencies
!pip install --upgrade pip >/dev/null
!pip install azure-storage-blob mysql-connector-python requests tqdm python-dotenv Pillow >/dev/null

In [ ]:
# Colab Cell 2 — MySQL connection and table setup
import os
from datetime import datetime
import mysql.connector
from mysql.connector import errorcode

MYSQL_HOST = os.getenv("MYSQL_HOST", "cycleai.mysql.database.azure.com")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", "admindev")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "Zana81333")
MYSQL_DB = os.getenv("MYSQL_DB", "cycle_ai")

def get_mysql_conn():
    cfg = {
        "host": MYSQL_HOST,
        "port": MYSQL_PORT,
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "database": MYSQL_DB,
        "autocommit": False,
    }
    try:
        conn = mysql.connector.connect(**cfg)
        return conn
    except mysql.connector.Error as err:
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("Something is wrong with your user name or password")
        elif err.errno == errorcode.ER_BAD_DB_ERROR:
            print("Database does not exist")
            raise

print("MySQL connected and table ensured: perception_poll_votes_ai")


In [ ]:
# Colab Cell 3 — Azure Blob connection and caching blob names
import os, io, random
from azure.storage.blob import BlobServiceClient
from PIL import Image

AZURE_CONN_STR = os.getenv("AZURE_CONN_STR", "")   # recommended: store in Colab secret or env
BLOB_CONTAINER = os.getenv("BLOB_CONTAINER", "london")

if not AZURE_CONN_STR:
    raise RuntimeError("AZURE_CONN_STR not set. Set as environment variable or inline in this cell.")

blob_svc = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

print("Listing blobs in 'Images/' folder (this may take a few seconds)...")

IMAGE_PREFIX = "training/training/Images/"
all_image_blobs = container_client.list_blobs(name_starts_with=IMAGE_PREFIX)

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp')

blob_names = [
    blob.name for blob in all_image_blobs
    if blob.name.lower().endswith(IMAGE_EXTENSIONS)
]

print(f"Found {len(blob_names)} image files.")
if len(blob_names) < 2:
    raise RuntimeError(f"Need at least 2 blobs in container to sample pairs. Found: {len(blob_names)}")

def download_blob_to_pil(blob_name, timeout=30):
    """Download a blob and return it as a PIL Image."""
    blob_data = container_client.download_blob(blob_name, timeout=timeout).readall()
    img = Image.open(io.BytesIO(blob_data)).convert("RGB")
    return img

def sample_two_random_blobs():
    a, b = random.sample(blob_names, 2)
    return a, b

print("Azure blob helpers ready.")


In [ ]:
HF_MODEL = os.getenv("HF_MODEL", "lmms-lab/LLaVA-OneVision-1.5-4B-Instruct")

In [ ]:
import torch
from transformers import AutoProcessor, AutoModel
from PIL import Image
import requests

HF_MODEL = "lmms-lab/LLaVA-OneVision-1.5-4B-Instruct"
# HF_MODEL = "Qwen/Qwen2-VL-7B"

try:
    processor = AutoProcessor.from_pretrained(HF_MODEL, trust_remote_code=True)

    model = AutoModel.from_pretrained(
        HF_MODEL,
        dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    print(f"✅ Model {HF_MODEL} loaded successfully using AutoModel.")

except Exception as e:
    print(f"❌ Final attempt failed with error: {e}")
    print("If this error persists, the model may have a unique dependency or a non-standard loading script that requires manual adaptation based on the model's documentation.")

In [ ]:
# Colab Cell 4
import os, io, json, time, base64, hashlib, random, re
from tqdm import tqdm
from PIL import Image
import requests
import mysql.connector
from datetime import datetime
from time import sleep
import uuid
from IPython.display import display, HTML
import pandas as pd
# ---------- CONFIG ----------
HF_API_TOKEN = os.getenv("HF_TOKEN", "")
WORKER_ID = os.getenv("WORKER_ID", str(uuid.uuid4()))

N = 20
BATCH_COMMIT = 20
RATE_LIMIT_SECONDS = float(os.getenv("RATE_LIMIT_SECONDS", "0.1"))
RETRY_MAX = 5
SAMPLE_TIMEOUT = 30
DEFAULT_CONFIDENCE_ON_FAIL = 0.0

FULL_PROMPT_TEXT = (
    "You will see two images: Image A (first) and Image B (second).\n\n"
    "TASK: Decide which image looks SAFER for cyclists.\n\n"
    "Respond with ONE line only in the exact required format:\n"
    "[VOTE]: [Vote Number] - [short reason]\n\n"
    "Where Vote Number is:\n"
    "  0 - Both look equally safe\n"
    "  1 - Image A looks safer\n"
    "  2 - Image B looks safer\n\n"
    "Keep your reason short. DO NOT include any extra text.\n\n"
    "Examples (Use this format EXACTLY):\n"
    "[VOTE]: 1 - Image A has a dedicated bike lane.\n"
    "[VOTE]: 2 - Image B has fewer vehicles present.\n"
    "[VOTE]: 0 - Both show similar traffic conditions.\n"
    "\nNow decide which image looks safer."
)

vote_counts = {0: 0, 1: 0, 2: 0}
results = []

def pil_to_data_uri(pil_img, quality=85):
    b64 = pil_to_base64_jpeg(pil_img, quality=quality)
    return f"data:image/jpeg;base64,{b64}"

def display_pair(pil1, pil2, vote, reason, name1, name2, max_width=320):
    uri1 = pil_to_data_uri(pil1)
    uri2 = pil_to_data_uri(pil2)
    caption = f"<b>Vote:</b> {vote} &nbsp;&nbsp; <b>Reason:</b> {reason}<br><b>Files:</b> {name1} — {name2}"
    html = f"""
    <div style="display:flex; align-items:flex-start; gap:12px; margin-bottom:10px;">
      <div style="text-align:center;">
        <img src="{uri1}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>{name1}</small>
      </div>
      <div style="text-align:center;">
        <img src="{uri2}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>{name2}</small>
      </div>
      <div style="flex:1; padding-left:8px;">
        {caption}
      </div>
    </div>
    """
    display(HTML(html))


def build_llava_prompt(user_text, suffix_text, processor_obj):
    try:
        image_token = getattr(processor_obj, "image_token", "<image>")
    except Exception:
        image_token = "<image>"
    prompt = (
        f"Image A: {image_token}\n"
        f"Image B: {image_token}\n\n"
        f"{user_text.strip()}\n{suffix_text.strip()}"
    )
    return prompt

# ---------- UTILITIES ----------
def pil_to_base64_jpeg(pil_img, quality=85):
    buff = io.BytesIO()
    pil_img.save(buff, format="JPEG", quality=quality)
    return base64.b64encode(buff.getvalue()).decode("utf-8")

def idempotency_key(name1, name2):
    s = f"{name1}|{name2}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def upsert_vote(conn, rec):
    cur = conn.cursor()
    sql = """
    INSERT INTO perception_poll_votes_ai
      (image_name1, image_name2, vote, model_id, user_id, reason)
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    params = (rec["image_name1"], rec["image_name2"], rec["vote"], rec["model_id"], rec["user_id"], rec.get("reason"))
    cur.execute(sql, params)
    cur.close()

def extract_generated_text(outputs, inputs, tokenizer):
    """
    Given generate() outputs and the inputs dict produced by the processor,
    return only the newly generated text (slice off the prompt tokens).
    This handles both plain tensors and the wrapper with .sequences.
    """
    if hasattr(outputs, "sequences"):
        seq = outputs.sequences[0]
    else:
        seq = outputs[0]

    input_len = 0
    if isinstance(inputs, dict) and "input_ids" in inputs:
        input_len = inputs["input_ids"].shape[1]

    gen_ids = seq[input_len:]
    try:
        import torch
        if isinstance(gen_ids, torch.Tensor):
            gen_ids = gen_ids.tolist()
    except Exception:
        pass

    text = tokenizer.decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return text.strip()

def call_model_locally(prompt_text, pil1, pil2):
    """
    Returns the model's generated text for two PIL images (only new tokens, not the prompt).
    """
    composed_prompt = build_llava_prompt(prompt_text, "", processor)

    inputs = processor(
        images=[pil1, pil2],
        text=composed_prompt,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        name: tensor.to(model.device, dtype=model.dtype) if tensor.is_floating_point() else tensor.to(model.device)
        for name, tensor in inputs.items()
    }

    import torch
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=0,
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    text_output = extract_generated_text(outputs, inputs, processor.tokenizer)
    print("Raw model generated (trimmed):", repr(text_output[:400]))
    return text_output

# ---------- MAIN LOOP ----------
print(f"Starting main loop: N={N}, HF model={HF_MODEL}, worker={WORKER_ID}")
conn = get_mysql_conn()
insert_count = 0

try:
    blob_names
except NameError:
    from azure.storage.blob import BlobServiceClient
    IMAGE_PREFIX = "training/training/Images/"
    IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp')

    print(f"Connecting to Azure to list files starting with '{IMAGE_PREFIX}'...")

    blob_svc = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
    container_client = blob_svc.get_container_client(BLOB_CONTAINER)

    all_image_blobs = container_client.list_blobs(name_starts_with=IMAGE_PREFIX)

    blob_names = [
        blob.name for blob in all_image_blobs
        if blob.name.lower().endswith(IMAGE_EXTENSIONS)
    ]

    if not blob_names:
        raise RuntimeError(f"No image files found in the '{IMAGE_PREFIX}' directory of the container.")

    print(f"Found {len(blob_names)} image files to sample from.")

pbar = tqdm(range(N), desc="Model votes")
for i in pbar:
    name1, name2 = random.sample(blob_names, 2)

    pil1 = pil2 = None
    for attempt in range(RETRY_MAX):
        try:
            pil1 = download_blob_to_pil(name1)
            pil2 = download_blob_to_pil(name2)
            break
        except Exception as e:
            wait = 2 ** attempt
            print(f"Blob download error for pair ({name1},{name2}) attempt {attempt+1}: {e}; sleeping {wait}s")
            time.sleep(wait)
    if pil1 is None or pil2 is None:
        pbar.set_postfix(status="download_fail")
        continue

    # call model
    try:
        raw = call_model_locally(FULL_PROMPT_TEXT, pil1, pil2)
    except Exception as e:
        print("Model call failed, skipping:", e)
        pbar.set_postfix(status="model_fail")
        continue

    txt = str(raw)

    print("-" * 50)
    print(f"Model Output for ({name1.split('/')[-1]}, {name2.split('/')[-1]}):")
    print(txt)  # now this should be just the model's generation (a short line)
    print("-" * 50)

    match = re.search(r"^(?:\[VOTE\]\s*[:\-]\s*)?\s*([0-2])\s*[:\-]\s*(.*)$", txt.strip(), re.MULTILINE | re.IGNORECASE)
    if not match:
        print(f"No valid 0/1/2 vote parsed for pair ({name1}, {name2}). Raw model text (truncated): {txt[:200]}")
        continue

    vote_val = int(match.group(1))
    reason_text = match.group(2).strip()

    IMAGE_PREFIX = "training/training/Images/"
    clean_name1 = name1.replace(IMAGE_PREFIX, '', 1)
    clean_name2 = name2.replace(IMAGE_PREFIX, '', 1)

    rec = {
        "id": idempotency_key(clean_name1, clean_name2),
        "image_name1": clean_name1,
        "image_name2": clean_name2,
        "vote": int(vote_val),
        "model_id": HF_MODEL,
        "user_id": WORKER_ID,
        "reason": reason_text,
    }

    try:
        upsert_vote(conn, rec)
        insert_count += 1
    except Exception as e:
        print("DB insert failed for record:", e)
        conn.rollback()
        pbar.set_postfix(status="db_error")
        continue

    vote_counts[vote_val] = vote_counts.get(vote_val, 0) + 1
    results.append({
      "image1": clean_name1,
      "image2": clean_name2,
      "vote": vote_val,
      "reason": reason_text,
      "model_id": HF_MODEL,
      "user_id": WORKER_ID
    })

    try:
      display_pair(pil1, pil2, vote_val, reason_text, clean_name1.split('/')[-1], clean_name2.split('/')[-1])
    except Exception as e:
      print("Display failed:", e)

    if insert_count % BATCH_COMMIT == 0:
        try:
            conn.commit()
        except Exception as e:
            print("DB commit failed:", e)
            conn.rollback()

    pbar.set_postfix(last_vote=rec["vote"])

    time.sleep(RATE_LIMIT_SECONDS)

try:
    conn.commit()
except Exception:
    conn.rollback()
conn.close()
print("Done. Inserted/updated records:", insert_count)

if results:
    df = pd.DataFrame(results)
    display(df.head(50))
else:
    print("No successful results captured in 'results' list.")


In [ ]:
# # Colab Cell 4 - printing confidence
# import os, io, json, time, base64, hashlib, random, re
# from tqdm import tqdm
# from PIL import Image
# import requests
# import mysql.connector
# from datetime import datetime
# from time import sleep
# import uuid

# # ---------- CONFIG ----------
# HF_API_TOKEN = os.getenv("HF_TOKEN", "")
# WORKER_ID = os.getenv("WORKER_ID", uuid.uuid4())

# N = 10
# BATCH_COMMIT = 20
# RATE_LIMIT_SECONDS = float(os.getenv("RATE_LIMIT_SECONDS", "0.3"))
# RETRY_MAX = 5
# SAMPLE_TIMEOUT = 30
# DEFAULT_CONFIDENCE_ON_FAIL = 0.0

# FULL_PROMPT_TEXT = (
#     "You will see two images: Image A (first) and Image B (second).\n\n"
#     "TASK: Decide which image looks SAFER for cyclists.\n\n"
#     "Respond with ONE line only in the exact required format:\n"
#     "[VOTE]: [Vote Number] - [short reason]\n\n"
#     "Where Vote Number is:\n"
#     "  0 - Both look equally safe\n"
#     "  1 - Image A looks safer\n"
#     "  2 - Image B looks safer\n\n"
#     "Keep your reason short (≤6 words). DO NOT include any extra text.\n\n"
#     "Examples (Use this format EXACTLY):\n"
#     "[VOTE]: 1 - Image A has a dedicated bike lane.\n"
#     "[VOTE]: 2 - Image B has fewer vehicles present.\n"
#     "[VOTE]: 0 - Both show similar traffic conditions.\n"
#     "\nNow decide which image looks safer."
# )


# def build_llava_prompt(user_text, suffix_text, processor_obj):
#     try:
#         image_token = getattr(processor_obj, "image_token", "<image>")
#     except Exception:
#         image_token = "<image>"
#     prompt = (
#         f"Image A: {image_token}\n"
#         f"Image B: {image_token}\n\n"
#         f"{user_text.strip()}\n{suffix_text.strip()}"
#     )
#     return prompt

# # ---------- UTILITIES ----------
# def pil_to_base64_jpeg(pil_img, quality=85):
#     buff = io.BytesIO()
#     pil_img.save(buff, format="JPEG", quality=quality)
#     return base64.b64encode(buff.getvalue()).decode("utf-8")

# def idempotency_key(name1, name2):
#     s = f"{name1}|{name2}"
#     return hashlib.sha256(s.encode("utf-8")).hexdigest()

# def upsert_vote(conn, rec):
#     cur = conn.cursor()
#     sql = """
#     INSERT INTO perception_poll_votes_ai
#       (image_name1, image_name2, vote, model_id, user_id, reason)
#     VALUES (%s, %s, %s, %s, %s)
#     """
#     params = (rec["image_name1"], rec["image_name2"], rec["vote"], rec["model_id"], rec["user_id"], rec.get("reason"))
#     cur.execute(sql, params)
#     cur.close()

# def extract_generated_text(outputs, inputs, tokenizer):
#     """
#     Given generate() outputs and the inputs dict produced by the processor,
#     return only the newly generated text (slice off the prompt tokens).
#     This handles both plain tensors and the wrapper with .sequences.
#     """
#     if hasattr(outputs, "sequences"):
#         seq = outputs.sequences[0]
#     else:
#         seq = outputs[0]

#     input_len = 0
#     if isinstance(inputs, dict) and "input_ids" in inputs:
#         input_len = inputs["input_ids"].shape[1]

#     gen_ids = seq[input_len:]
#     try:
#         import torch
#         if isinstance(gen_ids, torch.Tensor):
#             gen_ids = gen_ids.tolist()
#     except Exception:
#         pass

#     text = tokenizer.decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
#     return text.strip()

# def call_model_locally(prompt_text, pil1, pil2):
#     """
#     Returns the model's generated text for two PIL images (only new tokens, not the prompt).
#     Also prints the confidence distribution over the 0/1/2 vote tokens.
#     """
#     composed_prompt = build_llava_prompt(prompt_text, "", processor)

#     inputs = processor(
#         images=[pil1, pil2],
#         text=composed_prompt,
#         return_tensors="pt",
#         padding=True
#     )

#     inputs = {
#         name: tensor.to(model.device, dtype=model.dtype) if tensor.is_floating_point() else tensor.to(model.device)
#         for name, tensor in inputs.items()
#     }

#     import torch
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=32,
#             do_sample=False,
#             temperature=0,
#             eos_token_id=processor.tokenizer.eos_token_id,
#             pad_token_id=processor.tokenizer.eos_token_id,

#             # --- ADDED FOR CONFIDENCE ---
#             output_scores=True,
#             return_dict_in_generate=True
#         )

#     # Decode the generated text
#     text_output = extract_generated_text(outputs, inputs, processor.tokenizer)

#     # --- START: CONFIDENCE CALCULATION & PRINTING ---
#     try:
#         # First, check if the output *starts* with the format we expect.
#         match = re.search(r"^\s*\[VOTE\]\s*:", text_output.strip(), re.IGNORECASE)

#         if match and hasattr(outputs, "scores"):
#             # Get the token IDs for the three vote choices
#             # We check both with and without a leading space, as tokenizers vary
#             vote_ids_nospace = [
#                 processor.tokenizer.encode("0", add_special_tokens=False)[-1],
#                 processor.tokenizer.encode("1", add_special_tokens=False)[-1],
#                 processor.tokenizer.encode("2", add_special_tokens=False)[-1]
#             ]
#             vote_ids_space = [
#                 processor.tokenizer.encode(" 0", add_special_tokens=False)[-1],
#                 processor.tokenizer.encode(" 1", add_special_tokens=False)[-1],
#                 processor.tokenizer.encode(" 2", add_special_tokens=False)[-1]
#             ]

#             # Get all generated token IDs
#             input_len = inputs["input_ids"].shape[1]
#             gen_ids = outputs.sequences[0, input_len:] # Shape: [num_generated_tokens]

#             # Find the *index* (step) where the vote token was generated
#             vote_token_index = -1
#             chosen_vote_token_id = -1

#             # Iterate to find the *first* token that is a vote
#             for i, token_id in enumerate(gen_ids):
#                 if token_id in vote_ids_nospace or token_id in vote_ids_space:
#                     vote_token_index = i
#                     chosen_vote_token_id = token_id.item()
#                     break

#             if vote_token_index != -1:
#                 # We found the step where the vote was decided.
#                 # Get the logits for all tokens at that step.
#                 logits_at_vote_step = outputs.scores[vote_token_index][0]

#                 # Determine if the model was choosing between space or no-space tokens
#                 if chosen_vote_token_id in vote_ids_space:
#                     competing_token_ids = vote_ids_space
#                 else:
#                     competing_token_ids = vote_ids_nospace

#                 # Get the logits for just our three competing vote tokens
#                 vote_logits = logits_at_vote_step[competing_token_ids]

#                 # Apply softmax to get probabilities
#                 vote_probs = torch.nn.functional.softmax(vote_logits, dim=0)

#                 print(f"\n--- Vote Confidences (at token step {vote_token_index}) ---")
#                 print(f"  Vote 0 ({processor.tokenizer.decode(competing_token_ids[0])!r}): {vote_probs[0].item()*100:.2f}%")
#                 print(f"  Vote 1 ({processor.tokenizer.decode(competing_token_ids[1])!r}): {vote_probs[1].item()*100:.2f}%")
#                 print(f"  Vote 2 ({processor.tokenizer.decode(competing_token_ids[2])!r}): {vote_probs[2].item()*100:.2f}%")
#                 print(f"----------------------------------\n")

#     except Exception as e:
#         print(f"Could not calculate confidence: {e}")
#     # --- END: CONFIDENCE CALCULATION & PRINTING ---

#     print("Raw model generated (trimmed):", repr(text_output[:400]))
#     return text_output

# # ---------- MAIN LOOP ----------
# print(f"Starting main loop: N={N}, HF model={HF_MODEL}, worker={WORKER_ID}")
# conn = get_mysql_conn()
# insert_count = 0

# try:
#     blob_names
# except NameError:
#     from azure.storage.blob import BlobServiceClient
#     IMAGE_PREFIX = "training/training/Images/"
#     IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp')

#     print(f"Connecting to Azure to list files starting with '{IMAGE_PREFIX}'...")

#     blob_svc = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
#     container_client = blob_svc.get_container_client(BLOB_CONTAINER)

#     all_image_blobs = container_client.list_blobs(name_starts_with=IMAGE_PREFIX)

#     blob_names = [
#         blob.name for blob in all_image_blobs
#         if blob.name.lower().endswith(IMAGE_EXTENSIONS)
#     ]

#     if not blob_names:
#         raise RuntimeError(f"No image files found in the '{IMAGE_PREFIX}' directory of the container.")

#     print(f"Found {len(blob_names)} image files to sample from.")

# pbar = tqdm(range(N), desc="Model votes")
# for i in pbar:
#     name1, name2 = random.sample(blob_names, 2)

#     pil1 = pil2 = None
#     for attempt in range(RETRY_MAX):
#         try:
#             pil1 = download_blob_to_pil(name1)
#             pil2 = download_blob_to_pil(name2)
#             break
#         except Exception as e:
#             wait = 2 ** attempt
#             print(f"Blob download error for pair ({name1},{name2}) attempt {attempt+1}: {e}; sleeping {wait}s")
#             time.sleep(wait)
#     if pil1 is None or pil2 is None:
#         pbar.set_postfix(status="download_fail")
#         continue

#     # call model
#     try:
#         raw = call_model_locally(FULL_PROMPT_TEXT, pil1, pil2)
#     except Exception as e:
#         print("Model call failed, skipping:", e)
#         pbar.set_postfix(status="model_fail")
#         continue

#     txt = str(raw)

#     print("-" * 50)
#     print(f"Model Output for ({name1.split('/')[-1]}, {name2.split('/')[-1]}):")
#     print(txt)  # now this should be just the model's generation (a short line)
#     print("-" * 50)

#     match = re.search(r"^\s*\[VOTE\]\s*:\s*([0-2])\s*-\s*(.*)$", txt.strip(), re.IGNORECASE)
#     if not match:
#         print(f"No valid 0/1/2 vote parsed for pair ({name1}, {name2}). Raw model text (truncated): {txt[:200]}")
#         continue

#     vote_val = int(match.group(1))
#     reason_text = match.group(2).strip()

#     IMAGE_PREFIX = "Images/"
#     clean_name1 = name1.replace(IMAGE_PREFIX, '', 1)
#     clean_name2 = name2.replace(IMAGE_PREFIX, '', 1)

#     rec = {
#         "id": idempotency_key(clean_name1, clean_name2),
#         "image_name1": clean_name1,
#         "image_name2": clean_name2,
#         "vote": int(vote_val),
#         "model_id": HF_MODEL,
#         "user_id": WORKER_ID,
#         "reason": reason_text,
#     }

#     try:
#         upsert_vote(conn, rec)
#         insert_count += 1
#     except Exception as e:
#         print("DB insert failed for record:", e)
#         conn.rollback()
#         pbar.set_postfix(status="db_error")
#         continue

#     if insert_count % BATCH_COMMIT == 0:
#         try:
#             conn.commit()
#         except Exception as e:
#             print("DB commit failed:", e)
#             conn.rollback()

#     pbar.set_postfix(last_vote=rec["vote"])

#     time.sleep(RATE_LIMIT_SECONDS)

# try:
#     conn.commit()
# except Exception:
#     conn.rollback()
# conn.close()
# print("Done. Inserted/updated records:", insert_count)
